In [1]:
import importlib
import traceback
import inspect
import os
import sys
import pandas as pd
import nltk


sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), "www", "services"))

In [2]:
df_openalex = pd.read_csv("test_openalex.csv")
df_pubmed = pd.read_csv("test_pubmed.csv")
print(f"OpenAlex ready: {df_openalex.shape[0]} rows, {df_openalex.shape[1]} columns")
print(f"PubMed ready: {df_pubmed.shape[0]} rows, {df_pubmed.shape[1]} columns")

OpenAlex ready: 50 rows, 25 columns
PubMed ready: 50 rows, 25 columns


In [3]:
import ast

# Fix list columns — CSV saves lists as strings
list_cols = ["AU", "AF", "C1", "CR", "DE", "ID"]
for col in list_cols:
    df_openalex[col] = df_openalex[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    df_pubmed[col] = df_pubmed[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Fix string columns — CSV saves empty strings as NaN
str_cols = ["UT", "DI", "PMID", "TI", "SO", "JI", "PY", "DT", "LA", "RP", "AB", "VL", "IS", "BP", "EP", "SR", "DB"]
for col in str_cols:
    df_openalex[col] = df_openalex[col].fillna("").astype(str)
    df_pubmed[col] = df_pubmed[col].fillna("").astype(str)

print("DataFrames ready")

DataFrames ready


In [4]:
# Change this for each function you test
MODULE = "functions.get_sourcesproduction"
FUNCTION = "field_by_year"


mod = importlib.import_module(MODULE)
importlib.reload(mod)

# Find the actual function name if you don't know it
for name, obj in inspect.getmembers(mod, inspect.isfunction):
    if obj.__module__ == mod.__name__:
        print(f"Function found: {name}")
        print(f"Arguments: {inspect.signature(obj)}")

Function found: get_sources_production
Arguments: (df, num_of_sources_production, occurences)


In [5]:
func = getattr(mod, "get_sources_production")

for source_name, df in [("OpenAlex", df_openalex), ("PubMed", df_pubmed)]:
    try:
        result = func(
            df.copy(),
            num_of_sources_production=10,
            occurences="absolute"
        )
        print(f"[PASS] {source_name} — {type(result)}")
    except Exception as e:
        print(f"[FAIL] {source_name}")
        print(traceback.format_exc())

Processing field: SO

Processing field: PY

[PASS] OpenAlex — <class 'tuple'>
Processing field: SO

Processing field: PY

[PASS] PubMed — <class 'tuple'>
